# Predict horizon classes from a PLS projection

Loads the saved surface and ordinal classifier and predicts the coarse
time-horizon class for every row of a CSV. Nothing is fitted here and nothing is
plotted -- see `projection_direciton.ipynb` for the fitting and the figures.

The input CSV needs `PLS1`, `PLS2`, `PLS3` and `reconstruction_residual_PC1..3`.
If it also carries `time_horizon_months`, the true class is shown alongside the
prediction and scored.

## Setup

In [7]:
# Reload edited modules automatically: the helpers under `vendor/utils/` change
# often, and without this an already-imported module keeps its stale copy for the
# life of the kernel (an ImportError for a function that plainly exists on disk).
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The saved models unpickle classes from `temporal_manifolds`; a minimal copy of
# that code lives in `vendor/` at the repo root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vendor").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "vendor") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "vendor"))

from utils.horizon_classes import HORIZON_CLASS_LABELS, horizon_class
from utils.ordinal_regression import load_ordinal_model
from temporal_manifolds.viz.extruded_surface import load_surface_model

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

Defaults match the artifacts written by `projection_direciton.ipynb`:
`OFFSET_DEGREE = 2` for the surface and `DEGREE = 3` for the classifier.

In [8]:
INPUT_CSV = REPO_ROOT / "data" / "task_sf_split_activation_pls_projection.csv"

OFFSET_DEGREE = 2       # extrusion profile degree, picks the surface artifact
DEGREE = 3              # classifier polynomial degree, picks the model artifact

SURFACE_PATH = (
    REPO_ROOT / "models"
    / f"ctype_only_activation_surface_PLS1-PLS2-by-t_extruded-PLS3_degree-{OFFSET_DEGREE}.joblib"
)
CLASSIFIER_PATH = (
    REPO_ROOT / "models"
    / f"ctype_only_horizon_class_ordinal_binary-decomposition_degree-{DEGREE}.joblib"
)

# Columns the surface needs, in order. The classifier's own feature order comes
# from its artifact, so it does not need to be repeated here.
COORDINATE_COLUMNS = ["PLS1", "PLS2", "PLS3"]
PARAMETER_SAMPLES = 4000   # density of the nearest-t search when projecting

## Load the models

In [9]:
surface, surface_metadata = load_surface_model(SURFACE_PATH)
classifier, classifier_features, classifier_metadata = load_ordinal_model(CLASSIFIER_PATH)

print("surface   :", SURFACE_PATH.name)
print("            offset degree", surface.offset_degree,
      "| t range", np.round(surface.training_parameter_bounds, 3),
      "| fit rmse", round(surface.metrics.get("geometric_rmse", float("nan")), 3))
print("classifier:", CLASSIFIER_PATH.name)
print("            features", classifier_features)
print("            classes ", classifier_metadata["classes"])

surface   : ctype_only_activation_surface_PLS1-PLS2-by-t_extruded-PLS3_degree-2.joblib
            offset degree 2 | t range [-0.314  1.164] | fit rmse 8.899
classifier: ctype_only_horizon_class_ordinal_binary-decomposition_degree-3.joblib
            features ['t', 'u', 'reconstruction_residual_PC1', 'reconstruction_residual_PC2', 'reconstruction_residual_PC3']
            classes  [0, 1, 2, 3, 4, 5, 6, 7, 8]


## Read the input

In [ ]:
raw = pd.read_csv(INPUT_CSV)
df = raw
df = df[~df["time_horizon_months"].isna()]
# df = df[~df["task"].isin(['write a one-sentence email reply', 'write a short story'])] 
# raw=df
# The classifier's features are the surface coordinates plus whatever else it was
# trained on; `t` and `u` are derived below, the rest must come from the file.
derived = {"t", "u"}
required = COORDINATE_COLUMNS + [c for c in classifier_features if c not in derived]
missing = [column for column in required if column not in raw.columns]
if missing:
    raise KeyError(f"{INPUT_CSV.name} is missing required column(s): {missing}")

usable = raw[required].notna().all(axis=1)
df = raw[usable].copy()
print(f"{len(raw)} row(s) read, {len(df)} usable, {int((~usable).sum())} dropped for missing values")
df.head()

2721 row(s) read, 2721 usable, 0 dropped for missing values


,sample_index,absolute_token_position,template_id,template_metadata.prompt_framing,template_metadata.output_format,task,task_metadata.task_family,task_metadata.difficulty,task_metadata.domain,task_metadata.complexity,...,time_horizon_months,source_sample_count,PLS1,PLS2,PLS3,log10_time_horizon_months,reconstruction_residual_rms,reconstruction_residual_PC1,reconstruction_residual_PC2,reconstruction_residual_PC3
0,0,-1,task_available_time,<averaged>,NaN,answer a yes-or-no question,quick_decision,low,communication,low,...,3.802570e-07,18,-127.588921,56.726726,-37.598733,-6.419923,0.515175,-10.639005,-8.504138,-10.475378
1,198,-1,task_available_time,<averaged>,NaN,choose between two lunch options,personal_choice,low,personal_lifestyle,low,...,3.802570e-07,18,-127.914819,59.498778,-0.142830,-6.419923,0.410602,-9.303996,-2.778227,-6.744665
2,396,-1,task_available_time,<averaged>,NaN,copy a short code from one screen to another,copy_information,low,administrative,low,...,3.802570e-07,18,-100.928384,54.356102,16.014012,-6.419923,0.322947,-10.046666,0.814609,0.428591
3,3190,-1,task_available_time,<averaged>,NaN,press a button when a light turns green,quick_action,low,personal_lifestyle,low,...,3.802570e-07,18,-91.208870,43.227942,14.258389,-6.419923,0.322384,-10.936632,2.625606,1.550258
4,0,-1,direct_help,NaN,NaN,answer a yes-or-no question,quick_decision,low,communication,low,...,3.802570e-07,24,-74.713188,36.398348,-31.792237,-6.419923,0.209747,5.972024,2.579082,-2.814801


## Predict

In [11]:
# Surface coordinates first: u is exactly PLS3, t is the nearest point along
# the extruded curve.
t, u, distance = surface.project(
    df[COORDINATE_COLUMNS].to_numpy(float), parameter_samples=PARAMETER_SAMPLES
)
df["t"] = t
df["u"] = u
df["surface_distance"] = distance

predicted = classifier.predict(df[classifier_features].to_numpy(float))
df["horizon_class_predicted"] = predicted
df["horizon_class_label_predicted"] = np.array(HORIZON_CLASS_LABELS, dtype=object)[predicted]

print("distance to surface: mean %.3f, max %.3f" % (distance.mean(), distance.max()))
outside = (t < surface.training_parameter_bounds[0]) | (t > surface.training_parameter_bounds[1])
print("points projecting outside the fitted t range:", int(outside.sum()))

distance to surface: mean 7.010, max 41.032
points projecting outside the fitted t range: 0


## Results

In [12]:
columns = [
    "t", "u", "surface_distance",
    "horizon_class_predicted", "horizon_class_label_predicted",
]

# Score the predictions when the file carries the ground truth.
if "time_horizon_months" in df.columns:
    df["horizon_class"] = horizon_class(df["time_horizon_months"])
    df["horizon_class_label"] = np.array(
        list(HORIZON_CLASS_LABELS) + ["< 1 second"], dtype=object
    )[np.where(df["horizon_class"] < 0, len(HORIZON_CLASS_LABELS), df["horizon_class"])]
    known = df["horizon_class"] >= 0
    error = (
        df.loc[known, "horizon_class_predicted"] - df.loc[known, "horizon_class"]
    ).abs()
    print("scored on %d row(s) with a known horizon:" % int(known.sum()))
    print("  accuracy            %.4f" % (error == 0).mean())
    print("  within-one accuracy %.4f" % (error <= 1).mean())
    print("  mean absolute error %.4f classes" % error.mean())
    columns = ["horizon_class", "horizon_class_label"] + columns

summary = df["horizon_class_label_predicted"].value_counts().reindex(
    HORIZON_CLASS_LABELS, fill_value=0
)
print("\npredicted class counts:")
print(summary.to_string())

df[columns]

scored on 2721 row(s) with a known horizon:
  accuracy            0.7659
  within-one accuracy 0.9813
  mean absolute error 0.2569 classes

predicted class counts:
horizon_class_label_predicted
second - minute     106
minute - hour       212
hour - day          167
day - week          286
week - month        237
month - year        699
year - decade       506
decade - century    202
century - +inf      306


,horizon_class,horizon_class_label,t,u,surface_distance,horizon_class_predicted,horizon_class_label_predicted
0,0,second - minute,-0.225368,-37.598733,4.632098,0,second - minute
1,0,second - minute,-0.260133,-0.142830,2.718701,0,second - minute
2,0,second - minute,-0.165823,16.014012,14.704039,0,second - minute
3,0,second - minute,-0.114045,14.258389,9.878728,0,second - minute
4,0,second - minute,-0.026393,-31.792237,5.867439,0,second - minute
...,...,...,...,...,...,...,...
2881,8,century - +inf,0.931129,13.010083,6.752708,8,century - +inf
2882,8,century - +inf,1.008796,3.356197,2.100207,8,century - +inf
2883,8,century - +inf,0.983647,12.043890,8.197077,8,century - +inf
2884,8,century - +inf,0.957018,5.292781,3.456922,8,century - +inf
